In [1]:
import importlib

import PINN_utils
from PINN_utils import *

In [2]:
importlib.reload(PINN_utils)
from PINN_utils import *

In [3]:
pde = PDE(lambda x: torch.clamp(0.9 - torch.sin(2.5 * x) * 0.7, max=0.9, min=0.2))
breaking_time = pde.get_breaking_time(torch.linspace(0.0, 1.0, 500))
breaking_point = pde.get_breaking_point(torch.linspace(0.0, 1.0, 500))
shock_characteristic_start = pde.get_shock_characteristic_start(torch.linspace(0.0, 1.0, 500))

In [4]:
N_colocacao = 15000
N_inicial = 1500
N_contorno = 1500
sampler = DomainSampler.create(N_colocacao, N_inicial, N_contorno, tmin=0.0, tmax=0.3)

In [5]:
batch_size = 2048
batch_inicial = batch_size // 7
batch_contorno = batch_size // 7
batch_colocacao = batch_size - batch_contorno - batch_inicial

PINN_model = RHPiecewise_PINN(2, 1, 16, 4, t0=breaking_time, x0=breaking_point)

n_iter = 10000
lr = 1e-2
optimizer = optim.Adam(itertools.chain(PINN_model.net_IC.parameters(), PINN_model.net_BC.parameters()), lr=lr)
optimizer_RH = optim.Adam(PINN_model.net_RH.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_iter, eta_min=1e-6)

# a cada quantas iterações vamos alterar os pontos usados
N_change_points = 75

ts_treino = torch.linspace(sampler.tmin, sampler.tmax, 500).reshape(-1, 1).requires_grad_(True)

In [6]:
for it in range(n_iter):

    # train sub/super-shock networks
    batch = sampler.sample_batch(batch_colocacao, batch_inicial, batch_contorno)
    descontinuidade = PINN_model.get_discontinuity(ts_treino)
    mask_domain = ((descontinuidade > sampler.xmin) & (descontinuidade < sampler.xmax)).flatten()
    mask_shock = mask_domain & (ts_treino < breaking_time).flatten()
    y_fake = PINN_model(batch.get_all())

    L_f = pde.pde_loss(batch.get_colocacao(), y_fake[batch.slice_colocacao])
    L_b1 = pde.IC_loss(batch.get_inicial(), y_fake[batch.slice_inicial])
    L_b2 = pde.BC_loss(batch.get_contorno(), y_fake[batch.slice_contorno])
    L_IM = pde.IM_loss(batch.get_colocacao(), y_fake[batch.slice_colocacao])
    L_C = pde.continuity_loss(ts_treino, descontinuidade, mask_shock, PINN_model.net_IC, PINN_model.net_BC)
    L_PINN = L_f + 10 * (L_b1 + L_b2) + 5 * L_IM + 5 * L_C
    
    optimizer.zero_grad()
    L_PINN.backward()
    optimizer.step()
    scheduler.step()
    
    # train shock detection network
    L_RH = pde.RH_loss(ts_treino, descontinuidade, mask_domain, PINN_model.net_IC, PINN_model.net_BC, breaking_time, shock_characteristic_start)
    
    optimizer_RH.zero_grad()
    L_RH.backward()
    optimizer_RH.step()

    if it % N_change_points == 0:
        sampler.resample(PINN_model, pde)
    
    if (it + 1) % (n_iter // 10) == 0:
        print(f"Iter {it+1}: L_PINN={L_RH.item():.10f}")

Iter 1000: L_PINN=0.2136988938
Iter 2000: L_PINN=0.1756002307
Iter 3000: L_PINN=0.0038023095
Iter 4000: L_PINN=0.0031633500
Iter 5000: L_PINN=0.0049204607
Iter 6000: L_PINN=0.0119777787
Iter 7000: L_PINN=0.0021170159
Iter 8000: L_PINN=0.0086600892
Iter 9000: L_PINN=0.0031933393
Iter 10000: L_PINN=0.0035823283


In [7]:
PINN_model.save("resultados/prob4.pth")